### import library

In [1]:
import torch
import torch.nn as nn
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split

torch.cuda.empty_cache()

### Set Seed

In [2]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed = 59
set_seed(seed)

### Read dataset

#### Download dataset

In [3]:
!pip install -q gdown
!gdown --id 1i0v-_Fbc65PsRyTyAAAMbD9AkwgeAtur

/usr/local/lib/python3.11/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1i0v-_Fbc65PsRyTyAAAMbD9AkwgeAtur
From (redirected): https://drive.google.com/uc?id=1i0v-_Fbc65PsRyTyAAAMbD9AkwgeAtur&confirm=t&uuid=1d6f010d-64e5-47b4-94ef-a390d8c3d463
To: /content/img_cls_weather_dataset.zip
100% 613M/613M [00:08<00:00, 71.6MB/s]


#### Unzip file

In [4]:
import zipfile

with zipfile.ZipFile("img_cls_weather_dataset.zip", 'r') as zip_ref:
    zip_ref.extractall("/content/dataset")  # This will extract into ./dataset

In [5]:
root_dir = 'dataset/weather-dataset/dataset'
classes = {
    label_idx: class_name \
        for label_idx, class_name in enumerate(sorted(os.listdir(root_dir)))
}

img_paths = []
labels = []
for label_idx, class_name in classes.items():
    class_dir = os.path.join(root_dir, class_name)
    for img_filename in os.listdir(class_dir):
        img_path = os.path.join(class_dir, img_filename)
        img_paths.append(img_path)
        labels.append(label_idx)

### Split traing dataset

In [6]:
val_size = 0.2
test_size = 0.125
is_shuffle = True

X_train, X_val, y_train, y_val = train_test_split(
    img_paths, labels,
    test_size=val_size,
    random_state=seed,
    shuffle=is_shuffle
)

X_train, X_test, y_train, y_test = train_test_split(
    X_train, y_train,
    test_size=test_size,
    random_state=seed,
    shuffle=is_shuffle
)

### Define dataset

In [7]:
class WeatherDataset(Dataset):
    def __init__(self, X, y, transform=None):
        self.transform = transform
        self.img_paths = X
        self.labels = y

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img_path = self.img_paths[idx]
        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]

def transform(img, img_size=(224, 224)):
    img = img.resize(img_size)
    img = np.array(img)[..., :3]
    img = torch.tensor(img).permute(2, 0, 1).float()
    normalized_img = img / 255.0
    return normalized_img

train_dataset = WeatherDataset(
    X_train, y_train,
    transform=transform
)

val_dataset = WeatherDataset(
    X_val, y_val,
    transform=transform
)

test_dataset = WeatherDataset(
    X_test, y_test,
    transform=transform
)

### Define dataloader

In [8]:
train_batch_size = 256
test_batch_size = 128

train_loader = DataLoader(
    train_dataset,
    batch_size=train_batch_size,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=test_batch_size,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=test_batch_size,
    shuffle=False
)

### Define model

#### ResNet architecture (18-layer)

In [9]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Conv2d(
            in_channels, out_channels,
            kernel_size=3, stride=stride, padding=1
        )

        self.batch_norm1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(
            out_channels, out_channels,
            kernel_size=3, stride=1, padding=1
        )

        self.batch_norm2 = nn.BatchNorm2d(out_channels)

        self.downsample = nn.Sequential()
        if stride != 1 or in_channels !=out_channels:
            self.downsample = nn.Sequential(
                nn.Conv2d(in_channels, out_channels,
                          kernel_size=1, stride=stride),
                nn.BatchNorm2d(out_channels)
            )
        self.relu = nn.ReLU()

    def forward(self, x):
        shortcut = x.clone()
        x = self.conv1(x)
        x = self.batch_norm1(x)
        x = self.relu(x)
        x = self.conv2(x)
        x = self.batch_norm2(x)
        x += self.downsample(shortcut)
        x = self.relu(x)

        return x

In [10]:
class ResNet(nn.Module):
    def __init__(self, resisdual_block, n_blocks_lst, n_classes):
        super(ResNet, self).__init__()
        self.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3)
        self.batch_norm1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU()
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        self.conv2 = self.create_layer(resisdual_block, 64, 64, n_blocks_lst[0], 1)
        self.conv3 = self.create_layer(resisdual_block, 64, 128, n_blocks_lst[1], 2)
        self.conv4 = self.create_layer(resisdual_block, 128, 256, n_blocks_lst[2], 2)
        self.conv5 = self.create_layer(resisdual_block, 256, 512, n_blocks_lst[3], 2)
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(512, n_classes)

    def create_layer(self, resisdual_block, in_channels, out_channels, n_blocks, stride):
        blocks = []
        first_block = resisdual_block(in_channels, out_channels, stride)
        blocks.append(first_block)

        for idx in range(1, n_blocks):
            block = resisdual_block(out_channels, out_channels, stride=1)
            blocks.append(block)

        block_sequential = nn.Sequential(*blocks)
        return block_sequential

    def forward(self, x):
        x = self.conv1(x)
        x = self.batch_norm1(x)
        x = self.maxpool(x)
        x = self.relu(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.conv4(x)
        x = self.conv5(x)
        x = self.avgpool(x)
        x = self.flatten(x)
        x = self.fc1(x)
        return x


#### Call model

In [11]:
n_classes = len(list(classes.keys()))
device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = ResNet(
    resisdual_block=ResidualBlock,
    n_blocks_lst=[2, 2, 2, 2],
    n_classes=n_classes
).to(device)



### Define training function

#### Define evaluation function

In [12]:
def evaluate(model, dataLoader, criterion, device):
    model.eval()
    correct = 0
    total = 0
    losses = []
    with torch.no_grad():
        for inputs, labels in dataLoader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            losses.append(loss.item())
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted==labels).sum().item()
    loss = sum(losses) / len(losses)
    acc = correct / total

    return loss, acc

#### Define fit function

In [13]:
def fit(
        model,
        train_loader,
        val_loader,
        criterion,
        optimizer,
        scheduler,
        device,
        epochs
):
    train_losses = []
    val_losses = []
    val_accs = []

    for epoch in range(epochs):
        batch_train_losses = []
        model.train()

        for idx, (inputs, labels) in enumerate(train_loader): # because enumrate starts from 0, we need idx
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            batch_train_losses.append(loss.item())

        train_loss = sum(batch_train_losses) / len(batch_train_losses)
        train_losses.append(train_loss)

        val_loss, val_acc = evaluate(
            model, val_loader,
            criterion, device
        )

        val_accs.append(val_acc)
        val_losses.append(val_loss)
        scheduler.step()

        print(f'EPOCH {epoch+1}:\t Train_loss: {train_loss:.4f}\t Val_loss: {val_loss:.4f}\t Val_acc: {val_acc:.4f}')

    return train_losses, val_losses, val_accs


#### Training

In [14]:
lr = 1e-2
epochs = 30

def lr_lambda(epoch, warmup_epochs=5, total_epochs=30,
              init_scale=0.1, min_scale=0.3):
    scale_range = 1.0 - min_scale

    if epoch < warmup_epochs:
        warmup_factor = epoch / warmup_epochs
        return init_scale + (1.0 - init_scale)* warmup_factor

    decay_factor = (total_epochs - epoch) / (total_epochs - warmup_epochs)
    return min_scale + scale_range*max(0.0, decay_factor)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    model.parameters(), lr=lr
)

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

train_losses, val_losses, val_accs = fit(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    scheduler,
    device,
    epochs
)

EPOCH 1:	 Train_loss: 1.7457	 Val_loss: 6.9120	 Val_acc: 0.1420
EPOCH 2:	 Train_loss: 1.3032	 Val_loss: 4.3441	 Val_acc: 0.3358
EPOCH 3:	 Train_loss: 1.1767	 Val_loss: 1.7070	 Val_acc: 0.5244
EPOCH 4:	 Train_loss: 1.1309	 Val_loss: 1.7464	 Val_acc: 0.5353
EPOCH 5:	 Train_loss: 1.1125	 Val_loss: 1.5272	 Val_acc: 0.5390
EPOCH 6:	 Train_loss: 1.0737	 Val_loss: 4.9361	 Val_acc: 0.3496
EPOCH 7:	 Train_loss: 1.0050	 Val_loss: 4.3416	 Val_acc: 0.2039
EPOCH 8:	 Train_loss: 0.9590	 Val_loss: 2.1426	 Val_acc: 0.3918
EPOCH 9:	 Train_loss: 0.9228	 Val_loss: 2.6117	 Val_acc: 0.4253
EPOCH 10:	 Train_loss: 0.9012	 Val_loss: 2.4027	 Val_acc: 0.4086
EPOCH 11:	 Train_loss: 0.8621	 Val_loss: 1.1287	 Val_acc: 0.6184
EPOCH 12:	 Train_loss: 0.7943	 Val_loss: 1.6478	 Val_acc: 0.5178
EPOCH 13:	 Train_loss: 0.7984	 Val_loss: 2.6752	 Val_acc: 0.4610
EPOCH 14:	 Train_loss: 0.7427	 Val_loss: 1.1224	 Val_acc: 0.6242
EPOCH 15:	 Train_loss: 0.6832	 Val_loss: 2.1096	 Val_acc: 0.5259
EPOCH 16:	 Train_loss: 0.6951	 Val

### Evaluation

In [15]:
val_loss, val_acc = evaluate(
    model,
    val_loader,
    criterion,
    device
)

test_loss, test_acc = evaluate(
    model,
    test_loader,
    criterion,
    device
)

print('Evaluation on val/test dataset')
print('Val accuracy:', val_acc)
print('Test accuracy: ', test_acc)


Evaluation on val/test dataset
Val accuracy: 0.7509104151493081
Test accuracy:  0.7307132459970888
